In [1]:
import pandas as pd
import os
import sys
import warnings
from tqdm.notebook import tqdm
warnings.filterwarnings("ignore")

module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)

from src.portfolio_selection_ga import optimize_markowitz
from src.performance_tracker import PerformanceTracker
from src.utils import get_combinations, add_noise, get_returns_df, find_weights

In [2]:
grid = get_combinations()
data = get_returns_df()

In [3]:
parameters = {
    'risk_aver': 0.75,
    'min_ret_percentile': 0.25,
    'max_var_perc': 0.75,
    'window': 24,
    'step': 4
}

In [4]:
df = find_weights(data, parameters)
df.to_csv(f"../data/train/{parameters['risk_aver']}_{parameters['min_ret_percentile']}_{parameters['max_var_perc']}_{parameters["window"]}_{parameters["step"]}.csv", index=False)

In [34]:
df = df.set_index("Date")

In [ ]:
data

In [44]:
data_weights = df["weights"]

In [50]:
data_returns.name = "data_returns"

In [51]:
data_weights.name = "data_weights"

In [ ]:
data_weights

In [56]:
data_returns= data_returns[data_returns.index >= "2018-10-28"]

In [59]:
df_t = pd.concat([data_returns, data_weights ], axis=1)

In [62]:
df_t.columns = ["returns", "weights"]

In [66]:
portfolio_returns = df_t.apply(lambda x: x["returns"]  @ x["weights"], axis=1)

In [ ]:
data_returns = data.apply(lambda x: list(x), axis=1)
df_t = pd.merge(data_returns, data_weights, left_index="Date", right_index="Date", how="inner")

In [ ]:
data.apply(lambda x: df["weights"] @ x, axis=1)

In [ ]:
df[]

In [ ]:
df["portfolio_return"]

In [ ]:
fim_treino = "2020-01-01"
fim_teste = "2023-04-09"

In [ ]:
# data[data.index < fim_treino]

In [ ]:
# naive=data[fim_treino:fim_teste].mean(axis=1)
PerformanceTracker(naive, naive, risk_free=0.06, period="weekly")()

In [ ]:
portfolio_returns

In [81]:
naive = data["2020-01-01":"2023-04-09"]

In [ ]:
portfolio_returns[portfolio_returns.index >= "2019-12-25"]

In [ ]:
naive

In [ ]:
portfolio_returns["2020-01-01":"2024-09-04"]

In [ ]:

PerformanceTracker(portfolio_returns["2020-01-01":"2024-09-04"], naive, risk_free=0.06, period="weekly")()

In [ ]:

PerformanceTracker(df["portfolio_return"], df["portfolio_return"], risk_free=0.06, period="weekly")()

In [ ]:
data_inicio_treino = "2018-11-04"
data_fim_treino = "2020-06-21"
data_fim_teste = "2023-04-09"
risk_free = 0.06

In [ ]:
# Naive
naive_returns_path = Path('.').resolve().parent.joinpath("data", "base_dados.xlsx")
naive_returns = pd.read_excel(naive_returns_path)
naive_returns = naive_returns.set_index("Date").pct_change().dropna().mean(axis=1)
naive_returns.index = pd.to_datetime(naive_returns.index)
naive_returns.name = "Naive"

In [ ]:
market_data = naive_returns[(naive_returns.index >= data_fim_treino)&(naive_returns.index <= data_fim_teste)]

In [ ]:
(1 + market_data).cumprod().plot()

In [ ]:
(1 + portfolio_returns).cumprod().plot()

In [ ]:
market_data

result = PerformanceTracker(data=market_data, 
                            mkt_ret=market_data, 
                            risk_free=0.06, 
                            alph_var=0.05, 
                            period="weekly")()
result

In [ ]:
market_data = market_data[market_data.index >= "2021-04-18"]

In [ ]:
result = PerformanceTracker(data=portfolio_returns, 
                            mkt_ret=market_data, 
                            risk_free=0.06, 
                            alph_var=0.05, 
                            period="weekly")()
result

In [ ]:
(1+data.mean(axis=1)).cumprod().plot()

In [ ]:
data_fim_treino = "2020-05-19"
data_fim_teste = "2023-04-09"

In [ ]:
data = data.loc[data_fim_treino:data_fim_teste]

In [ ]:
num_ast = data.shape[1]

prms = {
    "risk_aver": 0.75,
    "min_ret_percentile": 0,
    "max_var_perc": 0.75,
    "window": 24,
    "step": 4
}

weights_df = pd.DataFrame(columns=data.columns)

for i in tqdm(range(prms["window"], len(data), prms["step"])):
  hist_data = data.iloc[i-prms["window"]:i].copy()
  best_individual = optimize_markowitz(hist_data, prms["risk_aver"], prms["min_ret_percentile"], prms["max_var_perc"])
  weights_dict = pd.Series(best_individual, index=hist_data.columns).to_dict()
  weights_dict["Date"] = data.iloc[i - prms["window"]:i].index.max()
  # weights_dict["Date"] = data.iloc[i - prms["window"]:i+1].index.max()
  print(hist_data.index)
  print(data.iloc[i - prms["window"]:i+1].index.max())
  weights_df = weights_df.append(weights_dict, ignore_index=True)
  print(weights_dict)

weights_df = weights_df.set_index("Date")
dates_index = data.index[data.index > data.iloc[prms["window"]-1:].index.min()]
print(dates_index)
weights_df = weights_df.reindex(index=dates_index, method='ffill')

In [ ]:
portfolio_returns = (weights_df * data[data.index > data.iloc[prms["window"]-1:].index.min()]).sum(axis=1)

In [ ]:
(1+data.mean(axis=1)).cumprod().plot()

In [ ]:
(1 + portfolio_returns).cumprod().plot()

In [ ]:
portfolio_returns.mean()

In [ ]:
(1 + data[data.index > data.iloc[prms["window"]-1:].index.min()].mean(axis=1)).cumprod().plot()

In [ ]:
(1+data.mean(axis=1)).cumprod().plot()

In [ ]:
(1+portfolio_returns).cumprod().plot()

In [ ]:
portfolio_returns

In [ ]:
PerformanceTracker(portfolio_returns, market_data, risk_free, 0.05, "weekly")()

In [ ]:
PortfolioSelectionGurobi(data, risk_aver=0.5, norm_param)

In [ ]:
best_individual_gurobi = optimize_markowitz_gurobi(data=data, risk_aver=0.5, min_ret_percentile=0.75, max_var_perc=1)

In [ ]:
best_individual_gurobi

In [ ]:
best_individual_gurobi = optimize_markowitz_gurobi(data=data, risk_aver=0.5, min_ret_percentile=0.75, max_var_perc=1)

In [ ]:
results.append({
    "weights": weights_df.copy(),
    "params": prms,
    "portfolio_returns": (weights_df * data[data.index > data.iloc[prms["window"]-1:].index.min()]).sum(axis=1)
})

In [ ]:
market_data

In [ ]:
def rename_keys(string, dic):
    new_dict = {}
    for key in dic.keys():
        new_key = key + string
        new_dict[new_key] = dic[key]
    return new_dict

In [ ]:
for item in results:
    # Treino
    market_data = naive_returns[(naive_returns.index >= data_inicio_treino)&(naive_returns.index <= data_fim_treino)]
    data_returns = item["portfolio_returns"]
    data_returns = data_returns[(data_returns.index >= data_inicio_treino)&(data_returns.index <= data_fim_treino)]
    result = PerformanceTracker(data=data_returns, 
                                mkt_ret=market_data, 
                                risk_free=risk_free, 
                                alph_var=0.05, 
                                period="weekly")()
    result = rename_keys("_treino", result)
    item.update(result)
    result = PerformanceTracker(data=market_data,
                                mkt_ret=market_data, 
                                risk_free=risk_free, 
                                alph_var=0.05,
                                period="weekly")()
    result = rename_keys("_treino_mercado", result)
    item.update(result)
    
df = pd.DataFrame([item for item in results])

In [ ]:
def generate_rankings(df, nm_rank):
    # sharpe
    df = df.sort_values(by=f"sharpe_ratio_{nm_rank}", ascending=False).reset_index(drop=True)
    df[f"sharpe_ratio_{nm_rank}_rank"] = (df.index + 1)#/df.shape[0]
    # max drawdown
    df = df.sort_values(by=f"max_drawdown_{nm_rank}", ascending=False).reset_index(drop=True)
    df[f"max_drawdown_{nm_rank}_rank"] = (df.index + 1)#/df.shape[0]
    # alpha
    df = df.sort_values(by=f"alpha_{nm_rank}", ascending=False).reset_index(drop=True)
    df[f"alpha_{nm_rank}_rank"] = (df.index + 1)#/df.shape[0]
    # value at risk
    df = df.sort_values(by=f"expected_loss_weekly_95_{nm_rank}", ascending=False).reset_index(drop=True)
    df[f"expected_loss_weekly_95_{nm_rank}_rank"] = (df.index + 1)#/df.shape[0]
    # generating rank
    df[f"ranking_{nm_rank}"] = df[f"alpha_{nm_rank}_rank"] + df[f"sharpe_ratio_{nm_rank}_rank"] + df[f"max_drawdown_{nm_rank}_rank"] + df[f"expected_loss_weekly_95_{nm_rank}_rank"]
    return df 

In [ ]:
df = generate_rankings(df, "treino")

In [ ]:
best_parameter = df.sort_values("ranking_treino", ascending=True).iloc[0].params

In [ ]:
market_data = naive_returns[(naive_returns.index >= data_inicio_treino)&(naive_returns.index <= data_fim_treino)]
data_returns = df[df["params"].apply(lambda x: x == best_parameter)].portfolio_returns.iloc[0]
data_returns = data_returns[(data_returns.index >= data_inicio_treino)&(data_returns.index <= data_fim_treino)]
result = PerformanceTracker(data=data_returns, 
                            mkt_ret=market_data, 
                            risk_free=risk_free, 
                            alph_var=0.05, 
                            period="weekly")()
result = rename_keys("_treino", result)
result

In [ ]:
df

In [ ]:
data.mean()